# S5.M3-1 组合选择改为训练集交叉验证

**目的**：S5.M1 用单一验证集选组合，分不开 0.005 量级的差距，「并列取计算量最小」又偏向联邦部分很小的 M1（[DR-M5-012](../../../registry/decision_records/DR-M5-012.yaml) 复核）。本步只换选择方式，候选与各候选的配置都沿用 S5.M1（[DR-M5-013](../../../registry/decision_records/DR-M5-013.yaml)）。

**做法**：
- 候选：P（优化版）、P_M4、M1、P+M6、M1+M6、P_M4+M6，有非共同客户时加 M1x、M1x+M6；
- 只用全部训练集做交叉验证，测试集不参与选择；
- 折按数据结构：行独立的 9 个任务分层 5 折；Santander G3 按客户分组 5 折；mock 四档按月份滚动外推 3 折（每折只用更早的月份训练）；
- R1：交叉验证平均 AUROC 最高；R2：与 R1 的逐折配对差不低于 −1 倍标准误的候选中取计算量最小；
- 选出的组合在测试集上评一次（6 seed，两层 bootstrap，数据集内 Holm）。

**数据**：公开数据，**开发证据**，不代表汇丰效果。测试集在以往步骤中评过；逐行数据与打分只在私有目录。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m3_cv_combo.yaml"
m3 = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
BASE_PATH = REPO_ROOT / m3["base_config"]
cfg = yaml.safe_load(BASE_PATH.read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components" / n for n in ("method_screen.py", "method_screen_cv.py")]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("base config:", BASE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(BASE_PATH))
print("决策记录:", m3["decision_record"], "| step:", m3["step_id"], "|", m3["purpose"])
print("seed:", m3["seed"], "| seeds:", m3["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
for c in COMPONENTS:
    print("组件:", c.relative_to(REPO_ROOT), "sha256:" + sha(c))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
RUN = pathlib.Path(os.environ[cfg["run_dir_env"]])
CVD = pathlib.Path(os.environ[m3["cv_dir_env"]])
print("启动指纹（运行开始时记录）：")
print((CVD / "fingerprint_launch.txt").read_text(encoding="utf-8"))

config: modules/m5_modeling/configs/s5_m3_cv_combo.yaml sha256:6841f6884602
base config: modules/m5_modeling/configs/s5_m1_method_screen.yaml sha256:761bae66da90
决策记录: registry/decision_records/DR-M5-013.yaml | step: S5.M3 | 组合选择改为训练集交叉验证——单一验证集分不开 0.005 量级差距、并列取最省偏向 M1（DR-M5-012 复核）
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: ad1ad92（工作区含本步未提交改动）
组件: modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
组件: modules/m5_modeling/components/method_screen_cv.py sha256:5f7a6b1b108c
启动指纹（运行开始时记录）：
05e19e025d0b61c012c837d87b45007b4065342274af45d0cd00864a8539b053  modules/m5_modeling/components/method_screen.py
029ad331cd6a041b95c1b3e2930423f5e8554911a1efa4b9edf4aa444d68cfbb  modules/m5_modeling/components/method_screen_cv.py
761bae66da90db70c6fac9a6f9a703424c04975a5a6635b8497e528ee69e3a7e  modules/m5_modeling/configs/s5_m1_method_screen.yaml
6841f6884602749c7e49ad425d1984b486fdbdea52b2ffdd636c734e07a3cf11  modules/m5_modeling/configs/s5_m3_cv_combo.yaml
41153e0f115df861ebf0

## 1. 运行来源与复现

整批由仓库组件 `method_screen_cv` 直接运行（启动指纹见上一格）。下一格把两个小任务从头重跑，与批量运行的结果 JSON 逐项比较：交叉验证分数、选择、测试 AUROC 与配对差必须完全相同。

In [2]:
from modules.m5_modeling.components import method_screen_cv as CV

REPRO = ["uci_bank", "coil2000"]          # 两个小任务从头重跑，与批量运行的结果 JSON 逐项比较
tmp = pathlib.Path(tempfile.mkdtemp())
for n in REPRO:
    again = CV.run(n, cfg, m3, PRIV, RUN, {})
    saved = json.loads((CVD / (n + ".json")).read_text(encoding="utf-8"))
    again = json.loads(json.dumps(again, ensure_ascii=False))
    for k in ("cv", "choice", "test", "contrasts"):
        assert again[k] == saved[k], (n, k)
    print(n, "· 交叉验证、选择、测试 AUROC 与配对差和批量运行完全相同；R1 =", saved["choice"]["r1"], "，R2 =", saved["choice"]["r2"])

[14:56:10] uci_bank · 交叉验证（row，5 折）{'P': 0.7992, 'P_M4': 0.7992, 'M1': 0.8008, 'P+M6': 0.8005, 'M1+M6': 0.8005, 'P_M4+M6': 0.8005} → R1 M1，R2 M1，用时 285.2s


uci_bank · 交叉验证、选择、测试 AUROC 与配对差和批量运行完全相同；R1 = M1 ，R2 = M1


[15:06:02] coil2000 · 交叉验证（row，5 折）{'P': 0.773, 'P_M4': 0.773, 'M1': 0.7694, 'P+M6': 0.773, 'M1+M6': 0.7758, 'P_M4+M6': 0.773} → R1 M1+M6，R2 M1+M6，用时 441.1s


coil2000 · 交叉验证、选择、测试 AUROC 与配对差和批量运行完全相同；R1 = M1+M6 ，R2 = M1+M6


## 2. 每个任务的选择与测试结果

- `s5m1_*`：S5.M1 单一验证集选出的组合及其相对优化版 P 的测试差（直接引用 S5.M1，不重算）；
- `cv_r1` / `cv_r2`：交叉验证两个规则的选择；`*_test`：测试 AUROC（6 seed 均值）。

In [3]:
OUT = REPO_ROOT / m3["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
R = {n: json.loads((CVD / (n + ".json")).read_text(encoding="utf-8")) for n in m3["datasets"]}
cvrows, conrows, choice = [], [], []
for n, r in R.items():
    for c, v in r["cv"].items():
        cvrows.append({"dataset": n, "scheme": r["scheme"], "candidate": c, "cv_mean": float(np.mean(v)),
                       "folds": len(v), "diff_vs_r1": r["choice"]["paired_vs_r1"][c]["diff"],
                       "se_vs_r1": r["choice"]["paired_vs_r1"][c]["se"], "cost": r["costs"][c]})
    for c in r["contrasts"]:
        conrows.append({"dataset": n, "contrast": "%s − %s" % (c["a"], c["b"]),
                        "pick": r["choice"][c["a"][-2:].lower()], "diff": c["diff"], "ci_low": c["ci_low"],
                        "ci_high": c["ci_high"], "p": c["p"], "p_holm": c["p_holm"]})
    s1 = r["s5m1"]["combo_minus_p"]
    choice.append({"dataset": n, "scheme": r["scheme"], "folds": len(r["folds"]),
                   "s5m1_val_choice": r["s5m1"]["choice"], "s5m1_minus_p": s1["diff"],
                   "s5m1_ci_low": s1["ci_low"], "s5m1_ci_high": s1["ci_high"],
                   "cv_r1": r["choice"]["r1"], "cv_r2": r["choice"]["r2"],
                   "p_test": r["test"]["P"]["auroc"], "r1_test": r["test"]["CV_R1"]["auroc"],
                   "r2_test": r["test"]["CV_R2"]["auroc"]})
cvdf, condf, chdf = pd.DataFrame(cvrows), pd.DataFrame(conrows), pd.DataFrame(choice)
cvdf.to_csv(OUT / "cv_candidates.csv", index=False, float_format="%.5f")
condf.to_csv(OUT / "test_contrasts.csv", index=False, float_format="%.5f")
chdf.to_csv(OUT / "choices.csv", index=False, float_format="%.5f")
print("每个任务的选择（S5.M1 单一验证集 vs 交叉验证 R1 / R2）与测试 AUROC（6 seed）：")
print(chdf.round(cfg["round_auc"]).to_string(index=False))
print()
print("测试集配对差（两层 bootstrap；数据集内 4 项 Holm）：")
print(condf.round(cfg["round_auc"]).to_string(index=False))

每个任务的选择（S5.M1 单一验证集 vs 交叉验证 R1 / R2）与测试 AUROC（6 seed）：
         dataset scheme  folds s5m1_val_choice  s5m1_minus_p  s5m1_ci_low  s5m1_ci_high   cv_r1  cv_r2  p_test  r1_test  r2_test
        uci_bank    row      5              M1        0.0005      -0.0036        0.0047      M1     M1  0.8072   0.8078   0.8078
        coil2000    row      5              M1        0.0058      -0.0067        0.0191   M1+M6  M1+M6  0.7275   0.7365   0.7365
    orange_churn    row      5              M1       -0.0062      -0.0115       -0.0013    P+M6   P+M6  0.7305   0.7325   0.7325
   orange_upsell    row      5            P_M4       -0.0023      -0.0059        0.0012 P_M4+M6      P  0.8668   0.8671   0.8668
orange_appetency    row      5              M1       -0.0222      -0.0358       -0.0099    P+M6      P  0.8496   0.8499   0.8496
    santander_g3  group      5              M1        0.0028      -0.0003        0.0060   M1+M6  M1+M6  0.8748   0.8786   0.8786
  mock_overlap20   time      3            

## 3. 汇总：三种选择方式各有多少任务优于 / 劣于优化版

判据（DR-M5-013 · D-4，先于结果写定）：R1 劣于优化版的任务数少于 S5.M1 的 2 个，且优于的任务数不少于 S5.M1 的 3 个，才算「交叉验证更好」。

In [4]:
def tally(df, label):
    gain = df[df["ci_low"] > 0]
    loss = df[df["ci_high"] < 0]
    return {"选择方式": label, "优于优化版": len(gain), "劣于优化版": len(loss), "不显著": len(df) - len(gain) - len(loss),
            "优于的任务": "、".join(gain["dataset"])}


s1 = chdf.rename(columns={"s5m1_ci_low": "ci_low", "s5m1_ci_high": "ci_high"})
rows = [tally(s1, "S5.M1 单一验证集"),
        tally(condf[condf["contrast"] == "CV_R1 − P"], "交叉验证 R1（主规则）"),
        tally(condf[condf["contrast"] == "CV_R2 − P"], "交叉验证 R2（一倍标准误内取最省）")]
tab = pd.DataFrame(rows)
tab.to_csv(OUT / "tally.csv", index=False)
print(tab.to_string(index=False))
r1 = tab.iloc[1]
print("DR-M5-013 · D-4 判定：交叉验证 R1 劣于优化版 %d 个（S5.M1 为 %d），优于 %d 个（S5.M1 为 %d）→ %s" % (
    r1["劣于优化版"], tab.iloc[0]["劣于优化版"], r1["优于优化版"], tab.iloc[0]["优于优化版"],
    "交叉验证更好" if (r1["劣于优化版"] < tab.iloc[0]["劣于优化版"] and r1["优于优化版"] >= tab.iloc[0]["优于优化版"])
    else "未达到「更好」的判据"))

              选择方式  优于优化版  劣于优化版  不显著                                                                                   优于的任务
       S5.M1 单一验证集      3      2    9                                               mock_overlap80、mock_balanced、santander_tx
      交叉验证 R1（主规则）      6      0    8                santander_g3、mock_overlap80、mock_balanced、santander_tx、homecredit、fedads
交叉验证 R2（一倍标准误内取最省）      7      0    7 santander_g3、mock_overlap50、mock_overlap80、mock_balanced、santander_tx、homecredit、fedads
DR-M5-013 · D-4 判定：交叉验证 R1 劣于优化版 0 个（S5.M1 为 2），优于 6 个（S5.M1 为 3）→ 交叉验证更好


## 4. 读法

**判定**（DR-M5-013 · D-4，先于结果写定）：达到「交叉验证更好」。
- 交叉验证 R1：6 / 14 优于优化版 P，0 / 14 劣于；S5.M1 单一验证集为 3 / 14 优于、2 / 14 劣于。
- R2（一倍标准误内取最省）：7 / 14 优于，0 / 14 劣于。

**变化在哪里**：
- S5.M1 变差的两个任务（Orange 流失 −0.006、Orange 偏好 −0.022）都是因为单一验证集上 M1 与最优并列、按「最省」被选中；交叉验证改选 P+M6 或 P，变为不显著（+0.002 / +0.000）。
- 新增显著的三个任务：Santander G3（M1+M6，+0.004）、HomeCredit（P+M6，+0.004）、FedAds（P+M6，+0.006；R2 选 M1x+M6，+0.010）。
- mock 重合 80、mock 平衡、Santander 交易三个任务的选择与 S5.M1 相同，增益不变。

**选了什么**：R1 在 14 个任务里有 13 个选了带 M6 的组合（P+M6 8 个、M1+M6 3 个、P_M4+M6 2 个），只有 UCI 选了不带 M6 的 M1。R2 有 11 个带 M6，另有 2 个退回 P、1 个 M1。所以「I2 + I3 + M6」是跨任务最稳的默认组合，M1 / M1x 是按任务选的可选层。

**边界**：
- 各候选的配置仍来自 S5.M1 的单一验证集，交叉验证只在候选之间选。
- 各任务的测试集以往评过，结果为开发证据；不代表汇丰效果。
- 技术侧自评，未经法务复核（W-001）。